# cpo-pic, stage 3: the glass-to-silicon taper in 3D (eigenmode expansion)

Upload `shaped_te3d.json` (and later `linear_te3d.json`) next to this notebook, then run the cells one at a time.
**Only cell 3 spends credits.** Cell 2 prints the price first; stop there and decide.

Change `JOB` below to run the linear taper instead.

In [ ]:
JOB = "shaped_te3d"          # or "linear_te3d"

import tidy3d as td
from tidy3d import web
sim = td.EMESimulation.from_file(f"{JOB}.json")
print(sim.eme_grid_spec.num_cells, "cells,", sim.eme_grid_spec.mode_spec.num_modes, "modes per cell")
print("length sweep:", None if sim.sweep_spec is None else sim.sweep_spec.scale_factors.tolist())
sim.plot(z=1000)   # cross-section: silicon on the adhesive band, the big glass guide below

## Cell 2: upload and ask the price (spends nothing)

In [ ]:
task_id = web.upload(sim, task_name=f"cpo-pic {JOB} EME")
print("estimated FlexCredits:", web.estimate_cost(task_id))
# STOP HERE. Decide with the number in front of you.

## Cell 3: run and download (this spends the credits)

In [ ]:
web.start(task_id)
web.monitor(task_id)
data = web.load(task_id, path=f"{JOB}_result.hdf5")

## Cell 4: transferred power, glass mode in to silicon mode out

In [ ]:
import numpy as np
S21 = data.smatrix.S21.isel(f=0, mode_index_in=0, mode_index_out=0)   # glass TE in -> silicon TE out
scales = [1.0] if sim.sweep_spec is None else sim.sweep_spec.scale_factors.tolist()
for k, s in enumerate(scales):
    t = abs(S21.isel(sweep_index=k).values) ** 2 if "sweep_index" in S21.dims else abs(S21.values) ** 2
    print(f"taper length {2000*s:6.0f} um : transferred into the silicon {float(t):6.1%}")

## Cell 5: where did the light go? Column sums near 100% mean the mode basis was big enough.

In [ ]:
S21 = data.smatrix.S21.isel(f=0); S11 = data.smatrix.S11.isel(f=0)
k = list(sim.sweep_spec.scale_factors).index(1.0) if "sweep_index" in S21.dims else None
P21 = abs(S21.isel(sweep_index=k) if k is not None else S21) ** 2
P11 = abs(S11.isel(sweep_index=k) if k is not None else S11) ** 2
print("|S21|^2 at the true length: rows = output mode at port 2, columns = input mode at port 1 (first 4 x 4)")
print(np.round(P21.values[:4, :4], 4))
for m in (0, 1):
    print(f"input mode {m}: arrives in any port-2 mode {float(P21.values[:, m].sum()):.3f}, reflected {float(P11.values[:, m].sum()):.3f}, column sum {float(P21.values[:, m].sum() + P11.values[:, m].sum()):.3f}")
if k is not None:
    print("\nsweep, column sum for input mode 0 (should stay near 1.0 at every length):")
    for j, s in enumerate(sim.sweep_spec.scale_factors):
        tot = float((abs(S21.isel(sweep_index=j)) ** 2).values[:, 0].sum() + (abs(S11.isel(sweep_index=j)) ** 2).values[:, 0].sum())
        print(f"   taper length {2000*s:6.0f} um : {tot:.3f}")

Afterwards: download `<JOB>_result.hdf5` from the file browser on the left and put it in `chain/eme/` of the repository.